# Notebook 6 – CNN Architecture Experiments



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Experiment design
Vary one factor at a time while holding split, preprocessing, seed, optimizer, epochs, and hardware constant. Predict output shape with `H_out=floor((H+2P-K)/S)+1`. Compare max/average pooling or stride-2 convolution deliberately. Record measured parameter count and time; repeat candidates across seeds.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader,TensorDataset
import torch.nn.functional as F
d=load_digits(); x=torch.tensor(d.images,dtype=torch.float32).unsqueeze(1)/16; x=F.interpolate(x,(16,16),mode='bilinear',align_corners=False); y=torch.tensor(d.target,dtype=torch.long)
a,b,ya,yb=train_test_split(x,y,test_size=.2,random_state=11,stratify=y); tr=DataLoader(TensorDataset(a,ya),128,shuffle=True); va=DataLoader(TensorDataset(b,yb),256)
class Trial(nn.Module):
 def __init__(self,width=16,kernel=3,drop=.0,bn=False):
  super().__init__(); p=kernel//2; layers=[nn.Conv2d(1,width,kernel,padding=p)]
  if bn: layers.append(nn.BatchNorm2d(width))
  layers += [nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(width,width*2,kernel,padding=p)]
  if bn: layers.append(nn.BatchNorm2d(width*2))
  layers += [nn.ReLU(),nn.AdaptiveAvgPool2d((2,2))]; self.features=nn.Sequential(*layers); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(drop),nn.Linear(width*8,10))
 def forward(self,x): return self.head(self.features(x))
configs=[('baseline',16,3,0.,False),('more filters',32,3,0.,False),('larger kernel',16,5,0.,False),('BN + dropout',16,3,.3,True)]
for name,w,k,drop,bn in configs:
 torch.manual_seed(42)
 m=Trial(w,k,drop,bn).to(device); opt=torch.optim.Adam(m.parameters(),lr=1e-3); start=time.perf_counter()
 for _ in range(2):
  m.train()
  for xb,y in tr:
   xb,y=xb.to(device),y.to(device); opt.zero_grad(); loss=nn.functional.cross_entropy(m(xb),y); loss.backward(); opt.step()
 m.eval(); good=total=0
 with torch.no_grad():
  for xb,y in va: good+=(m(xb.to(device)).argmax(1).cpu()==y).sum().item(); total+=len(y)
 print(name,'params',sum(p.numel() for p in m.parameters()),'val_acc',round(good/total,3),'seconds',round(time.perf_counter()-start,2))

baseline params 6090 val_acc 0.242 seconds 0.54
more filters params 21386 val_acc 0.214 seconds 1.23
larger kernel params 14538 val_acc 0.439 seconds 0.73
BN + dropout params 6186 val_acc 0.311 seconds 0.74


## Required experiment table
| Experiment | Architecture | Parameters | Validation Accuracy | Training Time | Observation |
|---|---|---:|---:|---:|---|
| Baseline | width 16, kernel 3 | measured | measured | measured | reference |
| More filters | width 32 | measured | measured | measured | compare accuracy and cost |
| Larger kernel | kernel 5 | measured | measured | measured | compare receptive field and cost |
| Regularized | BN + dropout | measured | measured | measured | compare stability/generalization |

